# Taiwanese Bankruptcy MLP Retrain (Class-Weighted BCE)

**Goal.** Replace the baseline MLP checkpoint at
`experiments/data/taiwan_bankruptcy_10_40_50/taiwan_bankruptcy_mlp.pth` with one
that actually learns class 1, by retraining on the *existing* 40% training
partition with class-weighted Binary Cross-Entropy. The baseline model collapsed
to a constant `predict-class-0` function under the 30:1 class imbalance, which
made every targeted attack against it trivially fail (LowProFool, PermuteAttack).

**What is reused unchanged (do NOT regenerate).**

- `train_test_data.npz` — same `X_svd / y_svd / X_train / y_train / X_test / y_test`.
- `preprocessing.npz` — same `weights / bounds_min / bounds_max`.
- `clean_svd_basis.npz` — same SVD basis computed on the 10% holdout.
- `metadata.json` — same splits and class distributions.
- `experiments/outputs/noise_scoring/taiwan_bankruptcy_10_40_50/20260514_161236/` —
  the Score-1 and Score-2 noise rankings on `X_test` are model-independent
  (pure SVD on `X_test`), so they remain valid and reusable by the four
  vaccination notebooks.

**What changes.**

- `taiwan_bankruptcy_mlp.pth` is **overwritten** with the retrained weights.
- The previous baseline checkpoint is preserved as
  `taiwan_bankruptcy_mlp_baseline.pth` (first run only — never re-overwritten).

The architecture (`TaiwanBankruptcyNet`: Linear(94,64)→ReLU→Linear(64,32)→ReLU→
Linear(32,2)→Softmax) is identical to the baseline, so the four attack
notebooks load the new state dict without any code change.

## 1. Imports + paths

In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix

DATA_DIR = Path('../../data/taiwan_bankruptcy_10_40_50').resolve()
CKPT_PATH = DATA_DIR / 'taiwan_bankruptcy_mlp.pth'
BASELINE_BACKUP = DATA_DIR / 'taiwan_bankruptcy_mlp_baseline.pth'

RANDOM_STATE = 42
N_EPOCHS = 400  # full-batch + weighted BCE converges slower than baseline
LR = 1e-3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Data dir: {DATA_DIR}')
assert DATA_DIR.is_dir(), DATA_DIR
assert CKPT_PATH.exists(), f'baseline checkpoint missing: {CKPT_PATH}'

## 2. Load the existing split (no re-split, no rescale)

Pulling `X_train`, `y_train`, `X_test`, `y_test` directly from the on-disk
`train_test_data.npz` guarantees the test set is byte-identical to the one the
noise scores were computed on. We never touch `X_svd`/`X_test` and never call
`train_test_split` again.

In [ ]:
data = np.load(DATA_DIR / 'train_test_data.npz')
X_train = data['X_train'].astype(np.float32)
y_train = data['y_train'].astype(np.int64)
X_test = data['X_test'].astype(np.float32)
y_test = data['y_test'].astype(np.int64)

n_train, n_features = X_train.shape
n_test = X_test.shape[0]
n_train_pos = int((y_train == 1).sum())
n_train_neg = int((y_train == 0).sum())
n_test_pos = int((y_test == 1).sum())
n_test_neg = int((y_test == 0).sum())

print(f'X_train: {X_train.shape}, y_train balance: '
      f'class 0={n_train_neg} ({n_train_neg/n_train:.2%}), '
      f'class 1={n_train_pos} ({n_train_pos/n_train:.2%})')
print(f'X_test:  {X_test.shape}, y_test balance: '
      f'class 0={n_test_neg} ({n_test_neg/n_test:.2%}), '
      f'class 1={n_test_pos} ({n_test_pos/n_test:.2%})')

## 3. Verify the baseline collapse on the loaded test set

Sanity check: confirm the current `taiwan_bankruptcy_mlp.pth` predicts class 0
for every test sample. If this assertion ever fails, the file has already been
replaced and you should be careful before overwriting it again.

In [ ]:
class TaiwanBankruptcyNet(nn.Module):
    """Same architecture as the baseline. forward() handles both 1D and 2D
    inputs so the LowProFool single-sample call (`model(x)` with x.dim()==1)
    keeps working without changes.
    """

    def __init__(self, D_in):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(D_in, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 2), nn.Softmax(dim=-1),
        )

    def forward(self, x):
        if x.dim() == 1:
            return self.layer(x.unsqueeze(0)).squeeze(0)
        return self.layer(x)


baseline_ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
baseline = TaiwanBankruptcyNet(baseline_ckpt['D_in']).to(device)
baseline.load_state_dict(baseline_ckpt['model_state_dict'])
baseline.eval()

with torch.no_grad():
    base_preds = baseline(torch.FloatTensor(X_test).to(device)).argmax(dim=1).cpu().numpy()
u, c = np.unique(base_preds, return_counts=True)
baseline_pred_hist = dict(zip(u.tolist(), c.tolist()))
print(f'Baseline test prediction histogram: {baseline_pred_hist}')
print(f'Baseline test accuracy: {(base_preds == y_test).mean():.4%}')
print(f'Baseline class-1 recall: {((base_preds == 1) & (y_test == 1)).sum() / max(1, n_test_pos):.4%}')

## 4. Retrain with class-weighted BCE

We keep the same `Softmax(dim=-1) + BCELoss(input, one_hot_target)` interface as
the baseline so the existing attack code (`bce(model(x+r), target)`) is
unaffected. The only change is per-sample weighting of the loss using the
sklearn `class_weight='balanced'` formula:

$$w_c = \frac{n}{C \cdot n_c}$$

with $C=2$. For 30:1 imbalance this gives roughly $w_0 \approx 0.52$ and
$w_1 \approx 15.5$, i.e. each class-1 sample contributes ~30x as much loss as
each class-0 sample.

In [ ]:
class_weights = torch.tensor(
    [n_train / (2.0 * n_train_neg), n_train / (2.0 * n_train_pos)],
    dtype=torch.float32, device=device,
)
print(f'Class weights (balanced): w0={class_weights[0].item():.4f}, '
      f'w1={class_weights[1].item():.4f}, ratio={class_weights[1].item() / class_weights[0].item():.2f}x')

torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

model = TaiwanBankruptcyNet(n_features).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
bce_none = nn.BCELoss(reduction='none')

X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.LongTensor(y_train).to(device)
y_train_onehot = nn.functional.one_hot(y_train_t, num_classes=2).float()
sample_weights = class_weights[y_train_t]  # (n_train,)

print('\nTraining (full-batch, weighted BCE):')
model.train()
for epoch in range(N_EPOCHS):
    optimizer.zero_grad()
    output = model(X_train_t)
    per_element = bce_none(output, y_train_onehot)         # (n_train, 2)
    per_sample = per_element.sum(dim=1)                    # (n_train,)
    loss = (per_sample * sample_weights).sum() / sample_weights.sum()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0 or epoch == 0:
        with torch.no_grad():
            tr_pred = output.argmax(dim=1).cpu().numpy()
            tr_acc = (tr_pred == y_train).mean()
            tr_rec1 = ((tr_pred == 1) & (y_train == 1)).sum() / max(1, n_train_pos)
        print(f'  epoch {epoch+1:>3d}/{N_EPOCHS} | loss={loss.item():.4f} | '
              f'train_acc={tr_acc:.4f} | train_recall_class1={tr_rec1:.4f}')

model.eval()
print('Training complete.')

## 5. Evaluate the retrained model

In [ ]:
with torch.no_grad():
    train_preds = model(torch.FloatTensor(X_train).to(device)).argmax(dim=1).cpu().numpy()
    test_preds = model(torch.FloatTensor(X_test).to(device)).argmax(dim=1).cpu().numpy()

train_acc = float((train_preds == y_train).mean())
test_acc = float((test_preds == y_test).mean())

print(f'Train accuracy: {train_acc:.4%}')
print(f'Test  accuracy: {test_acc:.4%}  '
      f'(majority baseline = {max(n_test_neg, n_test_pos)/n_test:.4%})')
print()
print('Test classification report:')
print(classification_report(
    y_test, test_preds,
    target_names=['Non-Bankrupt (0)', 'Bankrupt (1)'],
    digits=4, zero_division=0,
))
print('Test confusion matrix (rows=true, cols=pred):')
print(confusion_matrix(y_test, test_preds))

n_correct_class0 = int(((test_preds == 0) & (y_test == 0)).sum())
n_correct_class1 = int(((test_preds == 1) & (y_test == 1)).sum())
n_correct = n_correct_class0 + n_correct_class1
print(f'\nCorrectly classified test samples: {n_correct}/{n_test}')
print(f'  by true class: 0 -> {n_correct_class0}, 1 -> {n_correct_class1}')
print('(These are the points each attack notebook will target via LowProFool / PermuteAttack.)')

## 6. Save the new checkpoint, back up the baseline

Order of operations:

1. If `taiwan_bankruptcy_mlp_baseline.pth` does *not* yet exist, copy the
   current (still-baseline) `taiwan_bankruptcy_mlp.pth` to it. This makes the
   first run idempotent and protects the original from being lost on re-runs.
2. Overwrite `taiwan_bankruptcy_mlp.pth` with the retrained weights and a
   metadata block that records the training method (`WeightedBCELoss`, class
   weights, epochs).

In [ ]:
if not BASELINE_BACKUP.exists():
    shutil.copy2(CKPT_PATH, BASELINE_BACKUP)
    print(f'Backed up original baseline to {BASELINE_BACKUP.name}')
else:
    print(f'Backup already exists at {BASELINE_BACKUP.name} (left untouched).')

torch.save({
    'model_state_dict': model.state_dict(),
    'architecture': 'TaiwanBankruptcyNet',
    'D_in': int(n_features),
    'hidden_layers': [64, 32],
    'n_classes': 2,
    'train_accuracy': train_acc,
    'test_accuracy': test_acc,
    'optimizer': 'Adam',
    'lr': LR,
    'epochs': N_EPOCHS,
    'loss': 'WeightedBCELoss(class_weight=balanced)',
    'class_weights': class_weights.detach().cpu().tolist(),
    'random_state': RANDOM_STATE,
    'trained_on': '40% partition of taiwan_bankruptcy_10_40_50 (unchanged X_train/y_train)',
    'notes': (
        'Retrained to correct majority-class collapse under 30:1 imbalance. '
        'Test set, SVD basis, preprocessing, and noise scores are unchanged.'
    ),
}, CKPT_PATH)
print(f'Saved retrained checkpoint to {CKPT_PATH}')

print('\nFiles in data dir:')
for p in sorted(DATA_DIR.iterdir()):
    print(f'  {p.name:42s} ({p.stat().st_size/1024:.1f} KB)')

## 7. Confirm the saved checkpoint loads identically

Round-trip the new file through `TaiwanBankruptcyNet` and check that test
predictions match — guards against any state-dict-key drift before the attack
notebooks consume the file.

In [ ]:
loaded = torch.load(CKPT_PATH, map_location=device, weights_only=False)
reload_model = TaiwanBankruptcyNet(loaded['D_in']).to(device)
reload_model.load_state_dict(loaded['model_state_dict'])
reload_model.eval()
with torch.no_grad():
    reload_preds = reload_model(torch.FloatTensor(X_test).to(device)).argmax(dim=1).cpu().numpy()
assert np.array_equal(reload_preds, test_preds), 'reloaded model disagrees with in-memory model'
print('Reload round-trip OK.')
print(f'Saved metadata.loss = {loaded["loss"]!r}')
print(f'Saved metadata.test_accuracy = {loaded["test_accuracy"]:.4%}')

## 8. Reusing this model in the four attack notebooks

No code changes are required in:

- `targeted_vaccination_by_tier_LPF_taiwan.ipynb`
- `targeted_vaccination_clean_basis_LPF_taiwan.ipynb`
- `targeted_vaccination_by_tier_permuteattack_taiwan.ipynb`
- `targeted_vaccination_clean_basis_permuteattack_taiwan.ipynb`

They already point to `taiwan_bankruptcy_mlp.pth` and define an identical
`BankruptcyNet` architecture, so they will pick up the retrained weights
automatically on next run. The on-disk noise scores under
`experiments/outputs/noise_scoring/taiwan_bankruptcy_10_40_50/20260514_161236/`
are still valid because they were computed from `X_test` only, and `X_test`
was not modified.

Re-run order remains:

1. `targeted_vaccination_by_tier_LPF_taiwan.ipynb` (generates the LPF
   adversarial-sample npz)
2. `targeted_vaccination_clean_basis_LPF_taiwan.ipynb`
3. `targeted_vaccination_by_tier_permuteattack_taiwan.ipynb` (generates the
   PermuteAttack adversarial-sample npz; check the smoke-test ETA first)
4. `targeted_vaccination_clean_basis_permuteattack_taiwan.ipynb`